# LangChain Agent 

#### Authenticate with AWS

In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv()
print(loaded)

#### Import metrics for evaluation

In [ ]:
import metrics
from metrics import *


for name in dir(metrics):
    val = getattr(metrics, name)
    if isinstance(val, list):
        print(f"{name}:")
        for m in val:
            print(f"  - {m}")
        print()


#### Choose the set of metrics for evaluation

In [ ]:
metrics = single_ag_metrics
print("You've chosen the following metrics for evaluating the single LangChain agent:")
for m in metrics:
    print(f"  - {m}")

In [ ]:
from uaef.data import parse_ground_truth_row
# Import UAEF
from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime
print("✓ UAEF imported successfully!")

## Create LangChain Agent

#### Step 1: Build a simple LangChain agent

In [ ]:
# Step 1: Build a simple LangChain agent
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.tools import tool
from langchain_aws import ChatBedrock
import boto3
from botocore.config import Config
from uaef.adapters import LangChainAdapter
from uaef.api import evaluate
from uaef.models import GroundTruth, ToolCall
from datetime import datetime

# Define a simple tool
@tool
def search(query: str) -> str:
    """Search the web for information."""
    # Simulate search results
    return f"Found 10 results for '{query}' on python.org"

# Setup Bedrock LLM
region_name = "us-east-1"
my_config = Config(
    region_name=region_name,
    signature_version='v4',
    retries={'max_attempts': 3, 'mode': 'standard'}
)
bedrock_runtime = boto3.client(service_name="bedrock-runtime", config=my_config)
bedrock_llm = ChatBedrock(
    client=bedrock_runtime,
    model_id="anthropic.claude-3-sonnet-20240229-v1:0",
    model_kwargs={"max_tokens": 1024, "temperature": 0.0}
)

# Bind tools to LLM and create the agent executor
tools = [search]
llm_with_tools = bedrock_llm.bind_tools(tools)


#### Step 2: Run the LangChain Agent

In [ ]:
# Step 2: Run the agent and capture output in LangChain format
user_input = "Search for Python tutorials"
messages = [HumanMessage(content=user_input)]
response = llm_with_tools.invoke(messages)

# Tool call execution
intermediate_steps = []
if response.tool_calls:
    for tc in response.tool_calls:
        # Execute the tool
        tool_result = search.invoke(tc["args"])
        intermediate_steps.append({
            "action": tc["name"],
            "action_input": tc["args"],
            "observation": tool_result
        })
    
    # Get final response 
    messages.append(response)
    from langchain_core.messages import ToolMessage
    for tc, step in zip(response.tool_calls, intermediate_steps):
        messages.append(ToolMessage(content=step["observation"], tool_call_id=tc["id"]))
    final_response = llm_with_tools.invoke(messages)
    final_output = final_response.content
else:
    final_output = response.content


#### Step 3: Use LangChain Adapter 

In [ ]:
# Step 3: Package as LangChain-style result for the adapter
langchain_result = {
    "inputs": {"input": user_input},
    "outputs": {"output": final_output},
    "intermediate_steps": [
        (
            {"tool": step["action"], "tool_input": step["action_input"]},
            step["observation"]
        )
        for step in intermediate_steps
    ] if intermediate_steps else [],
    "session_id": "lc_001"
}

adapter = LangChainAdapter()
agent_trace = adapter.transform_to_canonical(langchain_result)


print(f"✓ Transformed LangChain output to AgentTrace")
print(f"  Trace ID: {agent_trace.trace_id}")
print(f"  Messages: {len(agent_trace.messages)}")
print(f"  Tool Calls: {len(agent_trace.tool_calls)}")
print(f"\nAgent Response: {final_output}")


#### Step 4: Evaluate

In [ ]:
# Step 4: Evaluate sample query
ground_truth = GroundTruth(
    expected_output="Python tutorials for beginners",
    expected_tool_calls=[
        ToolCall(
            name="search",
            arguments={"query": "Python tutorials"},
            timestamp=datetime.utcnow()
        )
    ]
)

result = evaluate(
    trace=agent_trace,
    ground_truth=ground_truth,
    metrics=metrics
)

print(f"\nOverall Score: {result.overall_score:.2f}")
print(f"Passed: {'✓ Yes' if result.passed else '✗ No'}")

## Batch Evaluation
1. Modify `data/ground-truth.xlsx` with your Ground Truth questions and answers.
2. Run batch evaluation by sending queries from Ground Truth to the live LangChain agent. 

#### Load Ground Truth Q & A

In [ ]:
import pandas as pd
import json
from datetime import datetime, timezone
from uuid import uuid4

from uaef.api import evaluate, batch_evaluate
from uaef.models import GroundTruth, ToolCall, AgentTrace, Message
from uaef.models.message import MessageRole

# Load ground truth data from Excel
excel_path = "data/ground-truth.xlsx"
df = pd.read_excel(excel_path)

# Convert Excel rows to JSON records for inspection
gt_json = df.to_dict(orient="records")

print(f"✓ Converted {len(gt_json)} rows to JSON")
print(f"✓ Loaded {len(df)} ground truth q/a from {excel_path}")
print(f"  Columns: {list(df.columns)}")
print(f"\nFirst few rows:")
df.head()



#### Reuse LangChain

In [ ]:
# Reuse the LangChain agent and adapter from Step 2 — no agent recreation
from langchain_core.messages import ToolMessage

adapter = LangChainAdapter()
traces = []
ground_truths = []

for i, row in enumerate(gt_json):
    query, expected, context, expected_tools = parse_ground_truth_row(row)

    # Send query to the LangChain agent
    messages = [HumanMessage(content=query)]
    response = llm_with_tools.invoke(messages)

    # Execute any tool calls
    intermediate_steps = []
    if response.tool_calls:
        for tc in response.tool_calls:
            tool_result = search.invoke(tc["args"])
            intermediate_steps.append({
                "action": tc["name"],
                "action_input": tc["args"],
                "observation": tool_result
            })
        messages.append(response)
        for tc, step in zip(response.tool_calls, intermediate_steps):
            messages.append(ToolMessage(content=step["observation"], tool_call_id=tc["id"]))
        final_response = llm_with_tools.invoke(messages)
        final_output = final_response.content
    else:
        final_output = response.content

    # Package as LangChain result for the adapter
    langchain_result = {
        "inputs": {"input": query},
        "outputs": {"output": final_output},
        "intermediate_steps": [
            (
                {"tool": step["action"], "tool_input": step["action_input"]},
                step["observation"]
            )
            for step in intermediate_steps
        ] if intermediate_steps else [],
        "session_id": f"batch_lc_{i}"
    }

    trace = adapter.transform_to_canonical(langchain_result)
    traces.append(trace)
    ground_truths.append(GroundTruth(
        expected_output=expected,
        expected_tool_calls=expected_tools,
        context_documents=[context] if context else []
    ))

    label = f"'{query[:50]}...'" if len(query) > 50 else f"'{query}'"
    print(f"  [{i+1}/{len(gt_json)}] {label}")

print(f"\n✓ Ran {len(traces)} queries through the LangChain agent")

#### Run evaluation on the agent traces

In [ ]:
# Batch evaluate all traces
batch_results = batch_evaluate(
    traces=traces,
    ground_truths=ground_truths,
    metrics=metrics,
    max_workers=4
)

print(f"{'='*50}")
print(f"BATCH RESULTS — LANGCHAIN AGENT")
print(f"{'='*50}")
for i, r in enumerate(batch_results):
    status = "✓" if r.passed else "✗"
    print(f"  {status} Test {i+1}: {r.overall_score:.2f}")

avg = sum(r.overall_score for r in batch_results) / len(batch_results)
pr = sum(1 for r in batch_results if r.passed) / len(batch_results) * 100
print(f"\nAverage Score: {avg:.2f}")
print(f"Pass Rate: {pr:.1f}%")

#### Export batch evaluation results

In [ ]:
# Save evaluation results
from uaef.utils import save_metric_results

filepath = save_metric_results(batch_results, gt_json, prefix="langchain_batch_results")
